In [1]:
%load_ext autoreload
%autoreload 2

import sys, json, time
from pathlib import Path
from collections import Counter
import polars as pl

ROOT = Path.cwd()
while not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.features.prepare import apply_rules, find_uninformative, META, RE_YEAR
cfg = load_config()
PROC, REPORTS = ROOT / 'data' / 'processed', ROOT / 'reports'
print('ROOT =', ROOT)

ROOT = c:\Users\HOME\Downloads\da2-credit-risk


In [3]:
t0 = time.time()
F = pl.scan_parquet(PROC / 'features.parquet')
n_before = len([c for c in F.collect_schema().names() if c not in META])

P = apply_rules(F)
feats = [c for c in P.collect_schema().names() if c not in META]
print(f'Sau R2–R4: {n_before} → {len(feats)} feature')

r1 = find_uninformative(P, feats)
for k, v in r1.items():
    print(f'  R1 {k}: {len(v)}')
drop_r1 = sorted(set().union(*r1.values()))
print(f'R1 loại tổng cộng: {len(drop_r1)}')
(REPORTS / 'prepare_drop_r1.json').write_text(json.dumps(r1, indent=2, ensure_ascii=False))

P.drop(drop_r1).sink_parquet(PROC / 'model_table.parquet')
print(f'Ghi xong model_table.parquet ({time.time()-t0:.0f}s)')

# Kiểm tra
M = pl.scan_parquet(PROC / 'model_table.parquet')
sch = M.collect_schema()
mf = [c for c in sch if c not in META]
yrs = [c for c in mf if c.endswith('_yrs_ago')]
print(f'\n{M.select(pl.len()).collect().item():,} dòng × {len(sch)} cột ({len(mf)} feature)')
print('kiểu:', dict(Counter(str(sch[c]) for c in mf)))
print('còn cột năm/tháng lịch:', [c for c in mf if RE_YEAR.search(c) or 'datemonth_' in c])
print('nhóm tax__*:', [c for c in mf if c.startswith('tax__')])
if yrs:
    print('miền giá trị *_yrs_ago:')
    print(M.select([e for c in yrs for e in (pl.col(c).min().alias(f'{c}|min'),
                                             pl.col(c).max().alias(f'{c}|max'))]).collect()
           .row(0, named=True))
else:
    print('Không còn cột *_yrs_ago (R2 sửa 28/10: loại hẳn cột năm lịch)')

Sau R2–R4: 725 → 677 feature
  R1 empty_train: 22
  R1 empty_valid: 22
  R1 constant_train: 49
R1 loại tổng cộng: 50
Ghi xong model_table.parquet (20s)

1,526,659 dòng × 634 cột (627 feature)
kiểu: {'Float32': 468, 'String': 33, 'Boolean': 6, 'UInt32': 112, 'Int64': 8}
còn cột năm/tháng lịch: []
nhóm tax__*: ['tax__count', 'tax__employer_nuniq', 'tax__amount_max', 'tax__amount_mean', 'tax__amount_sum', 'tax__amount_max_to_mean']
Không còn cột *_yrs_ago (R2 sửa 28/10: loại hẳn cột năm lịch)


In [5]:
pl.Config(tbl_rows=30, tbl_cols=-1, tbl_width_chars=220, fmt_str_lengths=70)
print('Bỏ qua: kiểm tra năm âm của *_yrs_ago (21/10) không còn áp dụng vì R2 sửa 28/10 đã loại hẳn cột năm lịch')

# --- Code cũ (21/10), giữ lại làm lịch sử ---
# YRS = [c for c in mf if c.endswith('_yrs_ago')]
# TRM = M.filter(pl.col('split') == 'train')
#
# rows = []
# for c in YRS:
#     r = TRM.select(
#         n=pl.col(c).count(),
#         min=pl.col(c).min(),
#         max=pl.col(c).max(),
#         n_am=(pl.col(c) < 0).sum(),
#         n_bang_0=(pl.col(c) == 0).sum(),
#         dr_am=pl.col('target').filter(pl.col(c) < 0).mean(),
#         dr_bang_0=pl.col('target').filter(pl.col(c) == 0).mean(),
#         dr_duong=pl.col('target').filter(pl.col(c) > 0).mean(),
#     ).collect().row(0, named=True)
#     rows.append({'feature': c.replace('credit_bureau_', 'cb_'), **r})
# print(pl.DataFrame(rows).with_columns(ty_le_am=pl.col('n_am') / pl.col('n')))
#
# # Số hồ sơ Train có ít nhất một feature năm âm
# any_neg = TRM.select(pl.any_horizontal([pl.col(c) < 0 for c in YRS]).sum()).collect().item()
# print(f'\nHồ sơ Train có ít nhất một sự kiện được ghi sang năm sau quyết định: {any_neg:,}')

Bỏ qua: kiểm tra năm âm của *_yrs_ago (21/10) không còn áp dụng vì R2 sửa 28/10 đã loại hẳn cột năm lịch


In [7]:
import re
from src.features.prepare import AUDIT_COLS
mt = pl.scan_parquet(PROC / 'model_table.parquet')
sch = mt.collect_schema()
feats = [c for c in sch.names() if c not in META]
print('Tổng cột:', len(sch), '| Feature:', len(feats))
print('Còn sót:', [c for c in feats if c.endswith('_yrs_ago') or 'pmts_year_1139T' in c
                   or re.search(r'(^|_)sex(_|$)', c)])
print('Cột kiểm tra công bằng còn trong bảng:', all(c in sch.names() for c in AUDIT_COLS))

Tổng cột: 634 | Feature: 627
Còn sót: []
Cột kiểm tra công bằng còn trong bảng: True
